<img src="https://isomer-user-content.by.gov.sg/397/debea305-557f-462b-8c58-d21400ce1ec3/rp-logo.png" width="200" alt="Republic Polytechnic"/>

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/koayst-rplesson/C3669C-2026-03/blob/main/Lesson_17/L17.ipynb)

# Setup and Installation

You can run this Jupyter notebook either on your local machine or run it at Google Colab.

* For local machine, it is recommended to install Anaconda and create a new development environment called `c3669c`.
* Pip/Conda install the libraries stated below when necessary.
---

# <font color='red'>ATTENTION</font>

## Google Colab
- If you are running this code in Google Colab, **DO NOT** store the API Key in a text file and load the key later from Google Drive. This is insecure and will expose the key.
- **DO NOT** hard code the API Key directly in the Python code, even though it might seem convenient for quick development.
- You need to enter the API key at python code `getpass.getpass()` when ask.

## Local Environment/Laptop
- If you are running this code locally in your laptop, you can create a env.txt and store the API key there.
- Make sure env.txt is in the same directory of this Jupyter notebook.
- You need to install `python-dotenv` and run the Python code to load in the API key.

---
```
%pip install python-dotenv

from dotenv import load_dotenv

load_dotenv('env.txt')
openai_api_key = os.getenv('OPENAI_API_KEY')
```
---

## GitHub/GitLab
- **DO NOT** `commit` or `push` API Key to services like GitHub or GitLab.

# Lesson 17

- RAGAS is designed to evaluate RAG applications, which combine retrieval (fetching relevant information from knowledge bases) and generation (LLM synthesizing answers). RAGAS provides a clear methodology to evaluate and improve RAG workflows.
- RAGAS evaluates pipelines on multiple dimensions, such as retrieval accuracy, response quality, and semantic alignment between input and output.
- Hallucinations are common in LLMs, where generated outputs include fabricated or inaccurate information. RAGAS helps assess the factual alignment of responses with retrieved knowledge.

In [1]:
%%capture --no-stderr
%pip install --quiet -U langchain==1.2.15
%pip install --quiet -U langchain-openai==1.1.12
%pip install --quiet -U ragas==0.4.3

In [2]:
# langchain        1.2.15
# langchain-core   1.2.26
# langchain-openai 1.1.12
# openai           2.30.0
# ragas            0.4.3
# datasets         4.8.4

In [3]:
import getpass
import os

# setup the OpenAI API Key

# get OpenAI API key ready and enter it when ask
os.environ["OPENAI_API_KEY"] = getpass.getpass()

 ········


## Examples of RAG Evaluation Metrics Using RAGAS

[Reference 1](https://docs.ragas.io/en/stable/), 
[Reference 2](https://docs.ragas.io/en/stable/getstarted/rag_eval/),
[Reference 3](https://docs.ragas.io/en/latest/concepts/metrics/index.html), 
[Reference 4](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/)

In [4]:
from ragas.llms import llm_factory
from openai import AsyncOpenAI

# Setup LLM
client = AsyncOpenAI()
llm = llm_factory("gpt-4o-mini", client=client)

# Context Precision
Did we retrieve all the necessary information needed to answer the question?

**Source**: Ragas docs<br/>
Context Precision is a metric that evaluates the retriever's ability to rank relevant chunks higher than irrelevant ones for a given query in the retrieved context. Specifically, it assesses the degree to which relevant chunks in the retrieved context are placed at the top of the ranking.

In [5]:
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecision

# Create metric
scorer = ContextPrecision(llm=llm)

# Evaluate
result = await scorer.ascore(
    user_input = "Who is Harry Potter's best friend?",
    reference = "Harry Potter's best friend is Ron Weasley.",
    retrieved_contexts = [
    "Harry Potter's best friend is Ron Weasley.",
    "Hermione Granger is one of Harry Potter's close friends."
        ]
)
print(f"Context Precision Score: {result.value}")

# sample answer
# Context Precision Score: 0.9987

Context Precision Score: 0.9999999999


# Context Recall

**Source**: Ragas docs</br>
This metric measures how many of the relevant documents (or pieces of information) were successfully retrieved. It focuses on not missing important results. Higher recall means fewer relevant documents were left out. In short, recall is about not missing anything important.

In [6]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics.collections import ContextRecall

scorer = ContextRecall(llm=llm)

result = await scorer.ascore(
    user_input = "Who is Harry Potter's best friend?",
    reference = "Harry Potter's best friend is Ron Weasley.",
    retrieved_contexts = ["Ron Weasley is one of Harry Potter's closest friends."]
)
print(f"Context Recall Score: {result.value}")

# sample answer
# 1.0

Context Recall Score: 1.0


# Context Entities Recall
Are the key entities (people, places, dates, concepts) from the ground truth present in your retrieved context?

*Source*: Rags docs</br>
This metric gives the measure of recall of the retrieved context, based on the number of entities present in both reference and retrieved_contexts relative to the number of entities present in the reference alone. Simply put, it is a measure of what fraction of entities is recalled from reference. This metric is useful in fact-based use cases like tourism help desk, historical QA, etc. This metric can help evaluate the retrieval mechanism for entities, based on comparison with entities present in reference, because in cases where entities matter, we need the retrieved_contexts which cover them.

In [7]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextEntityRecall

# Create metric
scorer = ContextEntityRecall(llm=llm)

# Evaluate
result = await scorer.ascore(
    reference = "Harry Potter's best friend is Ron Weasley.",
    retrieved_contexts = ["Ron Weasley is one of Harry Potter's closest friends."]
)
print(f"Context Entity Recall Score: {result.value}")

# sample answer
# Context Entity Recall Score: 0.999999995

Context Entity Recall Score: 0.999999995


# Noise Sensitivity
How well does your system handle irrelevant or distracing information in the retrieved context?

**Source**: Ragas</br>
It measures how often a system makes errors by providing incorrect responses when utilizing either relevant or irrelevant retrieved documents. The score ranges from 0 to 1, with lower values indicating better performance. Noise sensitivity is computed using the user_input, reference, response, and the retrieved_contexts.

In [8]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import NoiseSensitivity

# Create metric
scorer = NoiseSensitivity(llm=llm)

# Evaluate
result = await scorer.ascore(
    user_input="Who is Harry Potter?",
    response="Harry Potter is a young wizard known for defeating Voldemort and attending Hogwarts School of Witchcraft and Wizardry.",
    reference="Harry Potter is a fictional wizard character from J.K. Rowling's book series. He is famous for surviving an attack by Voldemort as a baby and attending Hogwarts.",
    retrieved_contexts=[
        "Harry Potter is the main character in a fantasy book series by J.K. Rowling.",
        "He is a wizard who attends Hogwarts School of Witchcraft and Wizardry.",
        "Harry is known for surviving Voldemort's attack as a baby.",
        "The series follows his adventures in the wizarding world."
    ]
)
print(f"Noise Sensitivity Score: {result.value}")

# sample answer 
# Noise Sensitivity Score: 0.3333333333333333

Noise Sensitivity Score: 0.3333333333333333


# Faithfulness

Is the generated answer grounded in the retrieved context or is the model hallucinatings?

**Source**: Ragas docs</br>
It measures how factually consistent a response is with the retrieved context. It ranges from 0 to 1, with higher scores indicating better consistency.

A response is considered faithful if all its claims can be supported by the retrieved context.

In [9]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness

# Create metric
scorer = Faithfulness(llm=llm)

# Evaluate
result = await scorer.ascore(
    user_input = "Who is Harry Potter's best friend?",
    response = "Harry Potter's best friend is Ron Weasley.",
    retrieved_contexts = ["Ron Weasley is one of Harry Potter's closest friends."]
)
print(f"Faithfulness Score: {result.value}")

# sample answer
# Faithfulness Score: 1.0

Faithfulness Score: 1.0


# Response Relevancy
Is the generated answer actually addressing the question asked? Your model might generate a perfectly factual, well-written response about the wrong thing. Response Relevancy ensures the answer stays on topic.

If it can recreate the original question from the answer, the answer must have addressed the question.

In [10]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.metrics.collections import AnswerRelevancy

# Setup embeddings
embeddings = embedding_factory("openai", model="text-embedding-3-small", client=client)

# Create metric
scorer = AnswerRelevancy(llm=llm, embeddings=embeddings)

# Evaluate
result = await scorer.ascore(
    user_input="When was the first super bowl?",
    response="The first superbowl was held on Jan 15, 1967"
)
print(f"Answer Relevancy Score: {result.value}")


Answer Relevancy Score: 0.9164903031982948
